[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/09_sensors_and_estimation.ipynb)

# Tutorial 9: Sensors & State Estimation

Use MuJoCo's built-in sensors: IMU, force/torque, joint encoders, cameras.

In [ ]:
!pip install -q mujoco mediapy matplotlib

In [ ]:
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media

## Sensor Types

MuJoCo supports many sensor types:
- `jointpos`, `jointvel` — encoder readings
- `accelerometer`, `gyro` — IMU
- `force`, `torque` — wrench sensors
- `framepos`, `framequat` — body pose
- `touch` — contact normal force

In [ ]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1"/>
    
    <body name="pendulum" pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.03" mass="1"/>
      <site name="imu_site" pos="0 0 -0.25"/>
      <site name="tip_site" pos="0 0 -0.5"/>
    </body>
  </worldbody>
  
  <actuator>
    <motor joint="hinge" ctrlrange="-5 5"/>
  </actuator>
  
  <sensor>
    <!-- Joint sensors -->
    <jointpos name="encoder" joint="hinge"/>
    <jointvel name="tachometer" joint="hinge"/>
    
    <!-- IMU at midpoint -->
    <accelerometer name="accel" site="imu_site"/>
    <gyro name="gyro" site="imu_site"/>
    
    <!-- Tip position -->
    <framepos name="tip_pos" objtype="site" objname="tip_site"/>
    
    <!-- Actuator force -->
    <actuatorfrc name="motor_force" actuator="motor"/>
  </sensor>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

# List all sensors
print(f"Number of sensors: {model.nsensor}")
print(f"Total sensor dimensions: {model.nsensordata}")
print()
for i in range(model.nsensor):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_SENSOR, i)
    dim = model.sensor_dim[i]
    adr = model.sensor_adr[i]
    print(f"  {name}: dim={dim}, address={adr}")

## Reading Sensor Data

All sensor values are in `data.sensordata` (flat array).
Use `sensor_adr` and `sensor_dim` to index, or use named access:

In [ ]:
# Simulate with a bang-bang controller and record sensors
mujoco.mj_resetData(model, data)
data.qpos[0] = 0.5  # start at 0.5 rad

times = []
encoder_data = []
accel_data = []
gyro_data = []
tip_data = []

while data.time < 5.0:
    # Simple controller
    data.ctrl[0] = -2.0 * np.sign(data.qpos[0])
    mujoco.mj_step(model, data)
    
    times.append(data.time)
    
    # Read sensors by name
    encoder_data.append(data.sensor('encoder').data.copy())
    accel_data.append(data.sensor('accel').data.copy())
    gyro_data.append(data.sensor('gyro').data.copy())
    tip_data.append(data.sensor('tip_pos').data.copy())

encoder_data = np.array(encoder_data)
accel_data = np.array(accel_data)
gyro_data = np.array(gyro_data)
tip_data = np.array(tip_data)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

axes[0].plot(times, np.degrees(encoder_data))
axes[0].set_ylabel('Encoder (deg)'); axes[0].set_title('Joint Position')

axes[1].plot(times, accel_data)
axes[1].set_ylabel('m/s²'); axes[1].set_title('Accelerometer (x, y, z)')
axes[1].legend(['x', 'y', 'z'])

axes[2].plot(times, gyro_data)
axes[2].set_ylabel('rad/s'); axes[2].set_title('Gyroscope (x, y, z)')
axes[2].legend(['x', 'y', 'z'])

axes[3].plot(times, tip_data)
axes[3].set_ylabel('m'); axes[3].set_title('Tip Position (world frame)')
axes[3].legend(['x', 'y', 'z']); axes[3].set_xlabel('Time (s)')

for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()

## Adding Sensor Noise

MuJoCo supports sensor noise natively via `noise` attribute:

In [ ]:
xml_noisy = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <body pos="0 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.03" mass="1"/>
      <site name="imu" pos="0 0 -0.25"/>
    </body>
  </worldbody>
  <sensor>
    <jointpos name="clean" joint="hinge"/>
    <jointpos name="noisy" joint="hinge" noise="0.05"/>
    <gyro name="gyro_noisy" site="imu" noise="0.1"/>
  </sensor>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(xml_noisy)
d = mujoco.MjData(m)
d.qpos[0] = 1.0

clean, noisy = [], []
for _ in range(1000):
    mujoco.mj_step(m, d)
    clean.append(float(d.sensor('clean').data[0]))
    noisy.append(float(d.sensor('noisy').data[0]))

plt.figure(figsize=(10, 3))
plt.plot(clean[:200], label='Clean', linewidth=2)
plt.plot(noisy[:200], label='Noisy (σ=0.05)', alpha=0.7)
plt.ylabel('Joint angle (rad)'); plt.xlabel('Step')
plt.legend(); plt.grid(True); plt.title('Sensor Noise'); plt.show()